In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from src.model import FiLMResNet2In, flatten_last
from src.normalizer import RunningMeanStd
from src.envpacker import packenv, packenv_batch
from src.utils import transition_ability_batched, update_ability_history, _summarize_tensor
import torch
from torch import nn
import torch.nn.functional as F
from tqdm import tqdm
import wandb
import os
import pickle

/opt/miniconda3/envs/sml/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/miniconda3/envs/sml/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` 

In [3]:
# Training configs
AGENTS = 50     # number of agents
LEARNING_RATE = 1e-3
TRAINING_STEPS = 30000
BATCH_SIZE = 10
DISPLAY_STEP = 100 # For visualization
TRAIN_STEP_INTERVAL = 2 # Interval of steps between training episodes
LAMBDA_AUX = 1.0 # Loss weighting parameters
LAMBDA_FOC = 1.0
MAX_HISTORY_LEN = 50


# Bewley model parameters
theta = 1 # CRRA
beta = 0.975 # Discount factor
A = 1 # Technology parameter
alpha = 0.33 # Capital share of income
gamma = 2 # Inverse Frisch elasticity
########################################### (MiLF inputs)
r = 0.06 # Interest rate (Return to savings)
w = 1 # Wage rate (Return to labor)
delta = 0.04 # Depreciation rate of capital
TAX_PARAMS = {
    "tax_consumption": 0.065,          # Consumption tax (fixed)
    "tax_income": 0.5,                # Tax on labor income
    "income_tax_elasticity": 0.5,     # Elasticity of labor supply w.r.t. after-tax income
    "saving_tax_elasticity": 0.5,     # Elasticity of savings w.r.t. after-tax income
    "tax_saving": 0.5                 # Tax on interest income
}
###########################################
p = 2.2e-6
q = 0.99

# shock parameters
# log e' = rho_v * log e + sigma_v * epsilon, epsilon ~ N(0,1)
rho_v = 0.95 # persistence of ability shock
sigma_v = 0.2 # std of ability shock
v_bar = 1.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Experiment setup ---
EXP_NAME = "exp_transition"
RUN_NAME = "run"
# --- Directory structure ---
BASE_DIR = os.path.join("./checkpoints", EXP_NAME, RUN_NAME)
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
STATE_DIR  = os.path.join(BASE_DIR, "states")

os.makedirs(WEIGHT_DIR, exist_ok=True)
os.makedirs(STATE_DIR, exist_ok=True)
final_weight_path = os.path.join(WEIGHT_DIR, "model_final.pt")
final_state_path = os.path.join(STATE_DIR, "final_state.pkl")


In [4]:
state_dim = 2*AGENTS + 2 # two state variables for each agent + 2 individual variables
cond_dim = 5 # exogenous variables for all agents in all worlds(Batch)
model = FiLMResNet2In(state_dim=state_dim, cond_dim=cond_dim,
                        hidden_dim=128, num_res_blocks=3, output_dim=3, dropout=0.1).to(DEVICE)

In [5]:
# Load model
checkpoint = torch.load(final_weight_path, map_location="cuda" if torch.cuda.is_available() else "mps")
if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)
model.eval()

# Load state
with open("/Users/chenjinghe/Desktop/python-projects/SML-2/checkpoints/exp_transition/run/states/state_step_0.pkl", "rb") as f:
    state_dict_train = pickle.load(f)

with open(final_state_path, "rb") as f:
    state_dict_tail = pickle.load(f)